In teminal run
  `gcloud auth application-default login`


In [1]:
# pickle gets confused
# %load_ext autoreload
# %autoreload 2

In [2]:
from adam_core.rotational_period.utils import _query_rotational_period_photometry_inputs, get_rotational_period_photometry_inputs
from adam_core.rotational_period.types import RotationalPeriodPhotometry, FourierFullResult
from adam_core.rotational_period.high_order_fourier import run_complete_fourier, run_fourier_cached
import pyarrow as pa
import pyarrow.compute as pc
from adam_core.time import Timestamp
from astropy.time import Time
from adam_core.orbits.query.horizons import cache_or_query_rotational_period_inputs, RotationalPeriodInput
from typing import Optional
import quivr as qv
import time
from adam_core.rotational_period.tests.paper_data import PAPER_TABLE2, get_target_record_from_paper

/workspaces/adam_core/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-05 01:30:11,409	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
DATASET_ID = "asteroid_institute_mpc_replica"
PHOTOMETRY_CACHE_FILE = "../../../../data/rotper_photometry_cache.parquet"
ROTATIONAL_PERIOD_INPUTS_CACHE = "../../../../data/rotper_horizons_cache.parquet"

C1C2_RESULT_CACHE = "../../../../data/rotper_c1c2_cache.parquet"
TYPE2_RESULT_CACHE = "../../../../data/rotper_type2_cache.parquet"
HG12STAR_RESULT_CACHE = "../../../../data/rotper_hg12star_cache.parquet"


In [4]:
def run_all(object_id: str, stn: str):
    photometry = get_rotational_period_photometry_inputs(object_id, stn, PHOTOMETRY_CACHE_FILE, DATASET_ID)
    photometry = photometry.sort_by(["obs_time"])
    horizons = cache_or_query_rotational_period_inputs(object_id, stn, photometry.obs_time.mjd(), ROTATIONAL_PERIOD_INPUTS_CACHE)
    horizons = horizons.sort_by(["obs_time"])
    c1c2 = run_fourier_cached(object_id, photometry, horizons, kind=None, cache_file=C1C2_RESULT_CACHE)
    type2 = run_fourier_cached(object_id, photometry, horizons, kind=2, cache_file=TYPE2_RESULT_CACHE)
    hg12star = run_fourier_cached(object_id, photometry, horizons, kind=-1, cache_file=HG12STAR_RESULT_CACHE)
    target = get_target_record_from_paper(object_id)
    return qv.concatenate([c1c2, type2, hg12star, target])


In [5]:
# Warm up a small'ish table 
selection = ["2025 MA19", "2025 MA45", "2025 MA46", "2025 MC34", "2025 MD38", "2025 MD40", "2025 MM81"] 
results = []
for id in selection:
    results.append(run_all(id, "X05"))
all_data = qv.concatenate(results)

Got 429 records out of 2358
Got 120 records out of 2358
Got 291 records out of 2358
Got 308 records out of 2358
Got 427 records out of 2358
Got 393 records out of 2358
Got 390 records out of 2358


Read total 2358 records
Read total 7 records from ../../../../data/rotper_c1c2_cache.parquet
Got 1 records out of 7
Return cached result for 2025 MA19 method Fourier c1c2
Read total 7 records from ../../../../data/rotper_type2_cache.parquet
Got 1 records out of 7
Return cached result for 2025 MA19 method Fourier type2
Read total 7 records from ../../../../data/rotper_hg12star_cache.parquet
Got 1 records out of 7
Return cached result for 2025 MA19 method Fourier GH12*
Read total 2358 records
Read total 7 records from ../../../../data/rotper_c1c2_cache.parquet
Got 1 records out of 7
Return cached result for 2025 MA45 method Fourier c1c2
Read total 7 records from ../../../../data/rotper_type2_cache.parquet
Got 1 records out of 7
Return cached result for 2025 MA45 method Fourier type2
Read total 7 records from ../../../../data/rotper_hg12star_cache.parquet
Got 1 records out of 7
Return cached result for 2025 MA45 method Fourier GH12*
Read total 2358 records
Read total 7 records from ../../

In [6]:
import numpy as np
import pandas as pd
from IPython.display import HTML, display

df = all_data.to_dataframe()
df["runtime_s"] = df["runtime"] / 1e9

METHODS = ["Fourier c1c2", "Fourier type2", "Fourier GH12*"]
METHOD_SHORT = {"Fourier c1c2": "c1c2", "Fourier type2": "type2", "Fourier GH12*": "GH12*"}

METRICS = ["period_h", "amplitude", "elongation", "color_gr", "color_gi", "color_ri"]
METRIC_LABELS = {
    "period_h": "Period (h)",
    "amplitude": "Ampl (mag)",
    "elongation": "Elongation",
    "color_gr": "g−r",
    "color_gi": "g−i",
    "color_ri": "r−i",
}

# Half the paper's rounding step for each metric
TOLERANCES = {
    "amplitude": 0.05,   # 1 decimal place → ±0.05
    "elongation": 0.05,  # 1 decimal place → ±0.05
    "color_gr": 0.0051,   # 2 decimal places → ±0.005
    "color_gi": 0.0051,
    "color_ri": 0.0051,
}


def metric_tol(metric, t_val):
    if metric == "period_h":
        # paper rounds > 0.1 h to 1 decimal, ≤ 0.1 h to 3 decimals
        if t_val is None:
            return 0.05
        return 0.05 if t_val > 0.1 else 0.0005
    return TOLERANCES[metric]


def format_target(metric, val):
    """Format a paper target value using the same rounding the paper uses."""
    if val is None:
        return "—"
    if metric == "period_h":
        return f"{val:.3f}" if val <= 0.1 else f"{val:.1f}"
    if metric in ("amplitude", "elongation"):
        return f"{val:.1f}"
    if metric in ("color_gr", "color_gi", "color_ri"):
        return f"{val:.2f}"
    return f"{val:.4f}"


CSS = """
<style>
  .rp-report { font-family: sans-serif; font-size: 13px; }
  .rp-report h2 { font-size: 15px; background: #263238; color: #fff;
                  margin: 20px 0 4px; padding: 6px 12px; border-radius: 3px; }
  .rp-table { border-collapse: collapse; margin-bottom: 4px; }
  .rp-table th { background: #546e7a; color: #fff; padding: 5px 10px;
                 text-align: center; white-space: nowrap; }
  .rp-table th.lh { text-align: left; }
  .rp-table td { padding: 4px 10px; border: 1px solid #cfd8dc; text-align: right;
                 white-space: nowrap; }
  .rp-table td.lh { background: #f5f5f5; font-weight: 600; text-align: left; }
  .rp-table tr.target-row td { background: #e3f2fd; font-weight: 600; }
  .rp-table tr.target-row td.lh { background: #bbdefb; }
  .rp-table .err { font-size: 11px; color: #555; }
</style>
"""


def _bg(err, tol):
    if err is None or np.isnan(float(err)):
        return "#e0e0e0"
    a = abs(float(err))
    if a <= tol:
        return "#c8e6c9"
    elif a <= 2 * tol:
        return "#fff9c4"
    return "#ffcdd2"


def _obs_bg(n_obs, t_obs):
    if n_obs is None or t_obs is None:
        return "#e0e0e0"
    if n_obs == t_obs:
        return "#c8e6c9"
    if n_obs < t_obs:
        return "#fff9c4"
    return ""


def build_report(df):
    parts = [f'<div class="rp-report">{CSS}']
    parts.append(
        '<h1 style="font-family:sans-serif;font-size:18px;margin:8px 0">'
        "Rotational Period Method Comparison</h1>"
    )

    col_headers = (
        ["Method", "#obs"]
        + [METRIC_LABELS[m] for m in METRICS]
        + ["Runtime (s)"]
    )

    for obj_id in df["object_id"].unique():
        obj = df[df["object_id"] == obj_id]
        target_row = obj[obj["method"] == "Fourier target"]
        target = target_row.iloc[0] if len(target_row) else None
        method_rows = obj[obj["method"] != "Fourier target"]

        arc = float(method_rows["arc_days"].iloc[0]) if len(method_rows) else 0.0
        parts.append(
            f'<h2>{obj_id}'
            f' <span style="font-weight:normal;font-size:12px">{arc:.2f}-day arc</span>'
            f'</h2>'
        )

        parts.append('<table class="rp-table"><thead><tr>')
        for h in col_headers:
            cls = ' class="lh"' if h == "Method" else ""
            parts.append(f"<th{cls}>{h}</th>")
        parts.append("</tr></thead><tbody>")

        t_obs_n = (
            int(target["num_obs"])
            if (target is not None and pd.notna(target["num_obs"]))
            else None
        )

        # Target row — values printed with the paper's own rounding
        if target is not None:
            parts.append('<tr class="target-row">')
            parts.append('<td class="lh">Target (paper)</td>')
            parts.append(f'<td>{"—" if t_obs_n is None else t_obs_n}</td>')
            for metric in METRICS:
                tv = float(target[metric]) if pd.notna(target[metric]) else None
                parts.append(f"<td>{format_target(metric, tv)}</td>")
            parts.append("<td>—</td>")
            parts.append("</tr>")

        # One row per method — computed values kept at full precision
        for method in METHODS:
            row = obj[obj["method"] == method]
            if len(row) == 0:
                continue
            r = row.iloc[0]
            parts.append("<tr>")
            parts.append(f'<td class="lh">{METHOD_SHORT[method]}</td>')

            # #obs
            n_obs = int(r["num_obs"]) if pd.notna(r["num_obs"]) else None
            diff = (n_obs - t_obs_n) if (n_obs is not None and t_obs_n is not None) else None
            obs_bg = _obs_bg(n_obs, t_obs_n)
            diff_s = (
                f'<br><span class="err">({diff:+d})</span>'
                if diff is not None and diff != 0
                else ""
            )
            style = f' style="background:{obs_bg}"' if obs_bg else ""
            parts.append(f'<td{style}>{"—" if n_obs is None else n_obs}{diff_s}</td>')

            # Metric columns
            for metric in METRICS:
                tv = (
                    float(target[metric])
                    if (target is not None and pd.notna(target[metric]))
                    else None
                )
                tol = metric_tol(metric, tv)
                v = float(r[metric]) if pd.notna(r[metric]) else None
                err = (v - tv) if (v is not None and tv is not None) else None
                bg = _bg(err, tol)
                sign = "+" if err is not None and err > 0 else ""
                v_s = f"{v:.4f}" if v is not None else "—"
                err_s = (
                    f'<br><span class="err">({sign}{err:.3f})</span>'
                    if err is not None
                    else ""
                )
                parts.append(f'<td style="background:{bg}">{v_s}{err_s}</td>')

            # Runtime
            rt = float(r["runtime_s"]) if pd.notna(r["runtime_s"]) else None
            parts.append(f'<td>{"—" if rt is None else f"{rt:.1f}"}</td>')
            parts.append("</tr>")

        parts.append("</tbody></table>")

    parts.append("</div>")
    return "\n".join(parts)

In [7]:
report_html = build_report(df)

report_path = "../../../../data/rotper_comparison_report.html"
with open(report_path, "w") as f:
    f.write(f'<!DOCTYPE html><html><head><meta charset="utf-8"></head><body>{report_html}</body></html>')
print(f"Saved to {report_path}")

display(HTML(report_html))

Saved to ../../../../data/rotper_comparison_report.html


Method,#obs,Period (h),Ampl (mag),Elongation,g−r,g−i,r−i,Runtime (s)
Target (paper),447,8.9,1.0,2.1,0.46,0.55,0.08,—
c1c2,429(-18),8.8713(-0.029),1.0708(+0.071),2.2149(+0.115),0.4355(-0.025),0.5549(+0.005),0.1194(+0.039),102.0
type2,429(-18),8.8713(-0.029),1.1018(+0.102),2.2665(+0.167),0.4321(-0.028),0.5494(-0.001),0.1173(+0.037),97.5
GH12*,429(-18),8.8713(-0.029),1.0838(+0.084),2.2364(+0.136),0.4358(-0.024),0.5539(+0.004),0.1180(+0.038),96.1
Method,#obs,Period (h),Ampl (mag),Elongation,g−r,g−i,r−i,Runtime (s)
Target (paper),140,1.6,0.6,1.5,0.45,0.60,0.14,—
c1c2,120(-20),1.6330(+0.033),0.5290(-0.071),1.4480(-0.052),0.4455(-0.005),0.5782(-0.022),0.1327(-0.007),59.3
type2,120(-20),1.6330(+0.033),0.5825(-0.017),1.5032(+0.003),0.4303(-0.020),0.6015(+0.001),0.1712(+0.031),56.9
GH12*,120(-20),1.6330(+0.033),0.5727(-0.027),1.4929(-0.007),0.4509(+0.001),0.5808(-0.019),0.1299(-0.010),56.3
Method,#obs,Period (h),Ampl (mag),Elongation,g−r,g−i,r−i,Runtime (s)
